
# Practical 25: **CV Studio - Multi‑Task Computer Vision Playground**  
This practical turns your course into a **single, unified app** where you can try many CV tasks from the series, side‑by‑side, in an interactive **Gradio** interface.

## What you can do
- **Classification** (ImageNet Top‑5): *ResNet50 & EfficientNet‑B0*
- **Object Detection**: *YOLOv8n*
- **Instance Segmentation**: *YOLOv8n‑seg*
- **Style Transfer / Fun Filters**: *OpenCV stylization, pencil‑sketch*
- **Model Comparison**: compare predictions across backbones




# Gradio: What, Why, and How

**Gradio** is a Python library that lets you turn models and CV pipelines into **interactive web apps** with just a few lines of code. 
That means your classmates (or future you!) can **upload an image/video, tweak sliders, press buttons, and see results instantly**, no need to write HTML/JS.

## Key Concepts

- **Components:** UI building blocks like `gr.Image`, `gr.Video`, `gr.Slider`, `gr.Dropdown`, `gr.Textbox`, etc.  
- **Events:** What triggers your function, for example, `.click`, `.change`, `.submit`.  
- **Functions:** Your Python functions that **take inputs** (components) and **return outputs** (components).  
- **`gr.Interface`:** Quick way to wire *one function* to some inputs/outputs. Great for first prototypes.  
- **`gr.Blocks`:** A flexible layout system to build **multi-tab / multi-task** apps with columns, rows, and state.

> In this practical we use **Blocks** to weave multiple tasks (classification, detection, and segmentation) into **one coherent interface**.

## Typical Patterns You’ll Use

1. **Quick demo with `gr.Interface`:**
   ```python
   import gradio as gr

   def predict(img):
       # 1) preprocess -> 2) model -> 3) postprocess
       return result

   demo = gr.Interface(fn=predict, inputs=gr.Image(type="numpy"), outputs=gr.Image())
   demo.launch()
   ```

2. **Rich apps with `gr.Blocks`:**
   ```python
   import gradio as gr

   with gr.Blocks(title="CV Studio") as demo:
       with gr.Tab("Classification"):
           inp = gr.Image(type="numpy", label="Upload an image")
           out = gr.Label()
           btn = gr.Button("Predict")
           btn.click(fn=predict_cls, inputs=inp, outputs=out)

       with gr.Tab("Detection"):
           # ... more components and callbacks ...

   demo.launch()
   ```

3. **State & Queues (for longer tasks):**
   - Use `gr.State()` to keep variables across interactions (e.g., loaded model, cached features).
   - Use `demo.queue(max_size=20, concurrency_count=2)` before `launch()` to handle multiple users smoothly.

## Good UX Tips for CV Apps
- **Show examples** with `examples=[...]` to make testing easy.
- Validate inputs (image size, channels) and **display friendly errors**.
- For heavy models: show **progress/status** and consider smaller images or batch preloading.
- Keep consistent **pre-/post-processing** across tasks so students can connect the dots.

---


## Gradio Crash Course

If you want to **learn Gradio step by step** in a short time, check out this YouTube crash course:  
[Gradio Crash Course - Fastest way to build & share Machine Learning apps](https://www.youtube.com/watch?v=eE7CamOE-PA&pp=ygUHZ3JhZGlvIA%3D%3D)



## What else is like Gradio? 

| Tool | Best For | Learning Curve | Notes |
|---|---|---|---|
| **Streamlit** | Fast dashboards, data apps | Very gentle | Superb for quick ML demos; less event-granular than Blocks, but great widgets and layout primitives. |
| **FastAPI + Uvicorn** | Production APIs | Moderate | You build a REST API; pair with a frontend (React/Vue) or a lightweight template. Great performance, testing, auth. |
| **Hugging Face Spaces** | Hosting demos online | Easy | You can deploy your Gradio/Streamlit app for free. Integrates with HF datasets/models. |
| **Panel (HoloViz)** | Scientific dashboards | Moderate | Strong plotting integration, good for notebooks → apps. |
| **Voilà** | Turn Jupyter notebooks into apps | Easy | Executes notebook server-side and serves outputs as a web app (no code changes, but less interactive wiring than Blocks). |
| **NiceGUI** | Pythonic web UIs | Moderate | Full-stack-ish components, routing, and layouts with a clean API. |


> In this course we pick **Gradio** because it's: (1) super-fast to prototype, (2) expressive enough for **multi-task** CV, and (3) easy to share.


---
To keep it fresh, test with **MS COCO 2017 sample images** or **Open Images** samples (any random street/indoor scenes). We won't train; we'll **infer** with pre‑trained weights.  
You can also use **your own photos** to see how models behave in the wild.



---
## 0) Setup & Installs
If you're on Colab/Kaggle, just run the pip cells. 


In [ ]:
# ----------------------------------------------
# Gradio UI Cell — explained for students:
# - Components define the UI (Image/Video/Buttons/etc).
# - Callbacks (e.g., btn.click) connect UI → Python functions.
# - Functions should: preprocess → run model → postprocess.
# - Keep I/O types consistent with gradio components.
# ----------------------------------------------

# Core CV & UI
# !pip install --upgrade pip
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  # or cpu wheel if needed
# !pip install ultralytics==8.3.7
# !pip install gradio==4.44.0 opencv-python==4.10.0.84 numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.0 pillow==10.4.0
# Optional: timm for extra backbones (ViT)
# !pip install timm==1.0.9



---
## 1) Utilities & Label Maps
We need an ImageNet label map for top‑5 class names. We'll download from `torchvision`'s built‑in metadata or fall back to a minimal list.


In [ ]:
# Context: This cell supports the CV pipeline (data/model/utils).
# Keep preprocessing ↔ postprocessing consistent across tasks.

import json, io, os, sys, math, time, tempfile
from typing import List, Tuple, Dict, Optional

import numpy as np
from PIL import Image

IMAGENET_IDX2NAME = {}
try:
    # Torchvision provides categories via weights meta (>=0.13)
    import torchvision as tv
    from torchvision.models import ResNet50_Weights, EfficientNet_B0_Weights
    IMAGENET_IDX2NAME = ResNet50_Weights.DEFAULT.meta.get('categories', {})
    if isinstance(IMAGENET_IDX2NAME, list):
        IMAGENET_IDX2NAME = {i:n for i,n in enumerate(IMAGENET_IDX2NAME)}
except Exception as e:
    print("Could not load ImageNet labels from torchvision. Using indices only.", e)
    IMAGENET_IDX2NAME = {i: f"class_{i}" for i in range(1000)}



---
## 2) Preprocessing helpers
Consistent resizing/normalization for classifiers; simple converters for detection/segmentation.


In [2]:
# Context: This cell supports the CV pipeline (data/model/utils).
# Keep preprocessing ↔ postprocessing consistent across tasks.

import torch
import torchvision.transforms as T
import cv2

def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

# Classifier transforms (match ImageNet training)
CLS_TRANSFORM_224 = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def pil_to_cv(img_pil: Image.Image):
    arr = np.array(img_pil.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

def cv_to_pil(img_cv):
    rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)



---
## 3) Model loaders
We load models **on first use** to keep memory manageable.


In [3]:
# Context: This cell supports the CV pipeline (data/model/utils).
# Keep preprocessing ↔ postprocessing consistent across tasks.

_classifier_cache = {}
_detector_cache = {}

def load_classifier(model_name: str, device=None):
    import torchvision.models as M
    device = device or get_device()
    if model_name in _classifier_cache:
        return _classifier_cache[model_name]

    if model_name == "resnet50":
        weights = M.ResNet50_Weights.DEFAULT
        model = M.resnet50(weights=weights).eval().to(device)
        preprocess = CLS_TRANSFORM_224
    elif model_name == "efficientnet_b0":
        weights = M.EfficientNet_B0_Weights.DEFAULT
        model = M.efficientnet_b0(weights=weights).eval().to(device)
        preprocess = CLS_TRANSFORM_224
    elif model_name == "vit_b_16":
        try:
            weights = M.ViT_B_16_Weights.DEFAULT
            model = M.vit_b_16(weights=weights).eval().to(device)
            preprocess = CLS_TRANSFORM_224
        except Exception as e:
            raise RuntimeError("ViT not available in your torchvision build.") from e
    else:
        raise ValueError("Unknown classifier: " + model_name)

    _classifier_cache[model_name] = (model, preprocess)
    return _classifier_cache[model_name]

def load_yolo(model_variant="yolov8n.pt"):
    # Uses ultralytics for detection/segmentation/tracking
    if model_variant in _detector_cache:
        return _detector_cache[model_variant]
    try:
        from ultralytics import YOLO
        model = YOLO(model_variant)
        _detector_cache[model_variant] = model
        return model
    except Exception as e:
        raise RuntimeError("Ultralytics not installed or model download failed. Run the pip cell above.") from e



---
## 4) Inference functions per mode
Each mode has a small, focused function. Results are returned in forms that the UI can render easily.


In [4]:
# Context: This cell supports the CV pipeline (data/model/utils).
# Keep preprocessing ↔ postprocessing consistent across tasks.

import torch
import torchvision as tv
import pandas as pd
import tempfile, os
import pandas as pd
from ultralytics import YOLO

def classify_top5(pil_img: Image.Image, backbones=("resnet50","efficientnet_b0")):
    device = get_device()
    rows = []
    vis_imgs = []  # keep original to display
    for name in backbones:
        model, preprocess = load_classifier(name, device=device)
        x = preprocess(pil_img).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        top5_idx = probs.argsort()[-5:][::-1]
        for rank, idx in enumerate(top5_idx, 1):
            rows.append({
                "model": name,
                "rank": rank,
                "class_id": int(idx),
                "class_name": IMAGENET_IDX2NAME.get(int(idx), f"class_{idx}"),
                "probability": float(probs[idx]),
            })
    df = pd.DataFrame(rows)
    return pil_img, df.sort_values(["model","rank"])

def detect_image(pil_img: Image.Image, conf=0.25):
    model = load_yolo("yolov8n.pt")
    res = model.predict(source=np.array(pil_img), conf=conf, verbose=False)
    annotated = res[0].plot()  # BGR numpy
    return cv_to_pil(annotated), res[0].tojson()

def segment_image(pil_img: Image.Image, conf=0.25):
    model = load_yolo("yolov8n-seg.pt")
    res = model.predict(source=np.array(pil_img), conf=conf, verbose=False, task="segment")
    annotated = res[0].plot()
    return cv_to_pil(annotated), res[0].tojson()

def fun_filters(pil_img: Image.Image, mode="stylize"):
    img = pil_to_cv(pil_img)
    if mode == "stylize":
        out = cv2.stylization(img, sigma_s=60, sigma_r=0.07)
    elif mode == "pencil":
        out_gray, out_color = cv2.pencilSketch(img, sigma_s=60, sigma_r=0.07, shade_factor=0.04)
        out = out_color
    else:
        out = img
    return cv_to_pil(out)



---
## 5) Gradio App (CV Studio)
Choose a mode from the sidebar, upload an image, and run. Results update dynamically.


In [ ]:
# ----------------------------------------------
# Gradio UI Cell — explained for students:
# - Components define the UI (Image/Video/Buttons/etc).
# - Callbacks (e.g., btn.click) connect UI → Python functions.
# - Functions should: preprocess → run model → postprocess.
# - Keep I/O types consistent with gradio components.
# ----------------------------------------------

import gradio as gr
import pandas as pd

with gr.Blocks(title="CV Studio — Multi‑Task CV Playground") as demo:
    gr.Markdown("# CV Studio — Multi‑Task CV Playground")
    gr.Markdown("Upload an image/video and choose a mode. Explore classification, detection, segmentation, filters, and model comparison.")

    with gr.Tabs():
        with gr.Tab("Classification"):
            gr.Markdown("**Top‑5 ImageNet predictions** with two backbones (ResNet50 & EfficientNet‑B0).")
            img_cls = gr.Image(type="pil", label="Upload image")
            btn_cls = gr.Button("Run classification")
            out_img_cls = gr.Image(label="Preview")
            out_tbl_cls = gr.Dataframe(label="Top‑5 per model")
            def _run_cls(img):
                if img is None: 
                    return None, pd.DataFrame()
                return classify_top5(img)
            btn_cls.click(_run_cls, inputs=img_cls, outputs=[out_img_cls, out_tbl_cls])

        with gr.Tab("Detection"):
            gr.Markdown("**YOLOv8** object detection on images.")
            img_det = gr.Image(type="pil", label="Upload image")
            conf_det = gr.Slider(0.1, 0.9, value=0.25, label="Confidence threshold")
            btn_det = gr.Button("Run detection")
            out_img_det = gr.Image(label="Detections")
            out_json_det = gr.Code(label="Raw JSON", language="json")
            def _run_det(img, conf):
                if img is None:
                    return None, "{}"
                vis, js = detect_image(img, conf=conf)
                return vis, js
            btn_det.click(_run_det, inputs=[img_det, conf_det], outputs=[out_img_det, out_json_det])

        with gr.Tab("Segmentation"):
            gr.Markdown("**YOLOv8‑seg** instance segmentation on images.")
            img_seg = gr.Image(type="pil", label="Upload image")
            conf_seg = gr.Slider(0.1, 0.9, value=0.25, label="Confidence threshold")
            btn_seg = gr.Button("Run segmentation")
            out_img_seg = gr.Image(label="Segmentations")
            out_json_seg = gr.Code(label="Raw JSON", language="json")
            def _run_seg(img, conf):
                if img is None:
                    return None, "{}"
                vis, js = segment_image(img, conf=conf)
                return vis, js
            btn_seg.click(_run_seg, inputs=[img_seg, conf_seg], outputs=[out_img_seg, out_json_seg])

        with gr.Tab("Fun Filters"):
            gr.Markdown("OpenCV **stylization** and **pencil‑sketch** for instant visual fun.")
            img_fun = gr.Image(type="pil", label="Upload image")
            mode_fun = gr.Radio(choices=["stylize","pencil"], value="stylize", label="Filter")
            btn_fun = gr.Button("Apply filter")
            out_fun = gr.Image(label="Result")
            def _run_fun(img, mode):
                if img is None:
                    return None
                return fun_filters(img, mode=mode)
            btn_fun.click(_run_fun, inputs=[img_fun, mode_fun], outputs=out_fun)

        with gr.Tab("Model Comparison"):
            gr.Markdown("Compare **ResNet50 vs EfficientNet‑B0** (and ViT if available) on the same image.")
            img_cmp = gr.Image(type="pil", label="Upload image")
            chk_res = gr.Checkbox(value=True, label="ResNet50")
            chk_eff = gr.Checkbox(value=True, label="EfficientNet‑B0")
            chk_vit = gr.Checkbox(value=False, label="ViT‑B/16 (if available)")
            btn_cmp = gr.Button("Compare")
            out_img_cmp = gr.Image(label="Preview")
            out_tbl_cmp = gr.Dataframe(label="Top‑5 per selected models")
            def _run_cmp(img, use_res, use_eff, use_vit):
                if img is None: 
                    return None, pd.DataFrame()
                backs = []
                if use_res: backs.append("resnet50")
                if use_eff: backs.append("efficientnet_b0")
                if use_vit: backs.append("vit_b_16")
                if not backs:
                    return img, pd.DataFrame()
                return classify_top5(img, backs)
            btn_cmp.click(_run_cmp, inputs=[img_cmp, chk_res, chk_eff, chk_vit], outputs=[out_img_cmp, out_tbl_cmp])

# To launch inside a notebook:
demo.launch(share=False)


c:\ProgramData\Anaconda3\envs\torch25\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


WARNING 'result.tojson()' is deprecated, replace with 'result.to_json()'.
WARNING 'result.tojson()' is deprecated, replace with 'result.to_json()'.


## Example: Running the Interface

After running the above code, you will see an interface like this:

![CV Studio Interface](images/CV_Studio_Interface.JPG)

When you upload an image and press **Run detection**, YOLOv8 highlights the objects:

![YOLOv8 Detection Result](images/detection.JPG)


Also, When you run the Gradio app code above, you’ll see a message like:
Running on local URL: for example (http://127.0.0.1:7860)
Click or copy this link into your browser, this is where your **CV Studio interface** will appear.


# Last Practical of the Series

This notebook is the **final practical** in our *Computer Vision Learning Coures*. By now you’ve built intuition and hands-on skill across:

1. **Foundations → CNNs & Training** (data pipelines, augmentation, hyper‑parameters)  
2. **Architectures** (AlexNet → VGG → ResNet → Inception → MobileNet → DenseNet → EfficientNet → ViT)  
3. **Tasks** (classification, detection, segmentation, tracking, metrics & evaluation)  
4. **MLOps Glue** (reproducibility, transfer learning, freezing vs fine‑tuning, experiment tracking)  
5. **Productization**, **this practical**: you shipped a **single multi‑task CV interface** students can run locally and extend.



#  A Journey Completed

Over this series of practicals, you’ve walked through the **entire landscape of Computer Vision**:  
from the first steps of CNNs, through world-class architectures, to building a **multi-task CV studio** that you can actually run and share.  

This wasn’t just about code. It was about **thinking like an engineer and a researcher**:  
- asking the right questions,  
- choosing the right tools,  
- experimenting, measuring, and improving,  
- and finally wrapping it all into something **others can use**.  
 

---

## What’s Next?
- Keep practicing, extend the app with new tasks (pose estimation, OCR, keypoint detection).  
- Share your demos with others, deploy to **Hugging Face Spaces** or **Streamlit Cloud**.  
- Stay curious, read new research papers, reproduce their models, and build your own CV playgrounds.  
- And most importantly, **teach others** what you’ve learned.  

---

*Remember: tools change, frameworks evolve, but the curiosity and creativity you built here will carry you through every new challenge in AI.*  

✨ Congratulations, you are now ready to explore the world as a true Computer Vision practitioner!  
